# Milestone 3

In [1]:
!pip install faiss-cpu

   ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
    --------------------------------------- 0.3/16.2 MB ? eta -:--:--
   -- ------------------------------------- 1.0/16.2 MB 3.9 MB/s eta 0:00:04
   ---- ----------------------------------- 1.8/16.2 MB 3.7 MB/s eta 0:00:04
   ------ --------------------------------- 2.6/16.2 MB 3.8 MB/s eta 0:00:04
   -------- ------------------------------- 3.4/16.2 MB 3.7 MB/s eta 0:00:04
   ---------- ----------------------------- 4.2/16.2 MB 3.6 MB/s eta 0:00:04
   ------------ --------------------------- 5.0/16.2 MB 3.7 MB/s eta 0:00:04
   -------------- ------------------------- 5.8/16.2 MB 3.7 MB/s eta 0:00:03
   ---------------- ----------------------- 6.6/16.2 MB 3.7 MB/s eta 0:00:03
   ------------------ --------------------- 7.3/16.2 MB 3.7 MB/s eta 0:00:03
   -------------------- ------------------- 8.1/16.2 MB 3.7 MB/s eta 0:00:03
   ---------------------- ----------------- 8.9/16.2 MB 3.7 MB/s eta 0:00:02
   ----------

In [2]:
import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

train = pd.read_csv('../dataset/train.csv') 

print("Creating knowledge base")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")

e:\IIT M DS\DL-GENAI Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Creating knowledge base
Loading embedding model and creating index


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2469.41it/s]


Knowledge base successfully created


In [3]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
row_150 = train.iloc[150] 
prompt_150 = str(row_150['prompt']) 
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 
ans_150 = str(row_150[row_150['answer']])

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 2290.60it/s]


In [4]:
result_q1 = zs(prompt_150, candidate_labels=labels_150, multi_label=False)
scores_dict = dict(zip(result_q1["labels"], result_q1["scores"]))
correct_score_q1 = scores_dict[ans_150]
 
print(f"Row 150 prompt: {prompt_150[:80]}...")
print(f"Correct answer text: {ans_150}")
print(f"All scores: {dict(zip(result_q1['labels'], [round(s,4) for s in result_q1['scores']]))}")
print(f"Predicted probability: {round(correct_score_q1, 3)}")

Row 150 prompt: Select the most accurate option: What is the butterfly effect, as defined by Lor...
Correct answer text: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
All scores: {'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."': 0.3844, 'The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."': 0.3787, 'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause subsequent states to differ greatly from the states

In [5]:
prompt_150_emb = model.encode([prompt_150], show_progress_bar=False)
prompt_150_emb = np.array(prompt_150_emb).astype("float32")
 
D, I = index.search(prompt_150_emb, k=10)
retrieved_indices = I[0].tolist()
 
print(f"Top 10 retrieved KB indices: {retrieved_indices}")
print(f"True document is at KB index: 150")
 
if 150 in retrieved_indices:
    rank_q2 = retrieved_indices.index(150) + 1
    print(f"Rank of true document: {rank_q2}")
else:
    print(f"True document NOT in top 10 retrieved results")
    rank_q2 = None

print("\nQ2 — Retrieved documents:")
for rank, idx in enumerate(retrieved_indices, 1):
    marker = " ← CORRECT" if idx == 150 else ""
    print(f"  Rank {rank:2d} | KB idx {idx:4d} | {kb[idx][:60]}...{marker}")

Top 10 retrieved KB indices: [663, 1701, 1269, 1532, 576, 847, 1693, 1906, 168, 150]
True document is at KB index: 150
Rank of true document: 10

Q2 — Retrieved documents:
  Rank  1 | KB idx  663 | The butterfly effect is the phenomenon that a small change i...
  Rank  2 | KB idx 1701 | The butterfly effect is the phenomenon that a small change i...
  Rank  3 | KB idx 1269 | The butterfly effect is the phenomenon that a small change i...
  Rank  4 | KB idx 1532 | The butterfly effect is the phenomenon that a small change i...
  Rank  5 | KB idx  576 | The butterfly effect is the phenomenon that a small change i...
  Rank  6 | KB idx  847 | The butterfly effect is the phenomenon that a small change i...
  Rank  7 | KB idx 1693 | The butterfly effect is the phenomenon that a small change i...
  Rank  8 | KB idx 1906 | The butterfly effect is the phenomenon that a small change i...
  Rank  9 | KB idx  168 | The butterfly effect is the phenomenon that a small change i...
  Rank 10 | KB idx

In [6]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

docs_10 = [kb[i] for i in retrieved_indices]

pairs     = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

reranked = sorted(zip(retrieved_indices, docs_10, ce_scores),
                  key=lambda x: x[2], reverse=True)
 
print(f"Reranked results (cross-encoder scores):")
rank_q3 = None
for rank, (kb_idx, doc, score) in enumerate(reranked, 1):
    marker = " ← CORRECT" if kb_idx == 150 else ""
    print(f"  Rank {rank:2d} | KB idx {kb_idx:4d} | score={score:.4f} | {doc[:50]}...{marker}")
    if kb_idx == 150:
        rank_q3 = rank
 
print(f"Rank of true document after cross-encoder reranking: {rank_q3}")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6666.49it/s]


Reranked results (cross-encoder scores):
  Rank  1 | KB idx  150 | score=4.7585 | The butterfly effect is the phenomenon that a smal... ← CORRECT
  Rank  2 | KB idx  847 | score=4.7526 | The butterfly effect is the phenomenon that a smal...
  Rank  3 | KB idx 1693 | score=4.7526 | The butterfly effect is the phenomenon that a smal...
  Rank  4 | KB idx 1906 | score=4.7526 | The butterfly effect is the phenomenon that a smal...
  Rank  5 | KB idx 1269 | score=4.7375 | The butterfly effect is the phenomenon that a smal...
  Rank  6 | KB idx 1532 | score=4.7375 | The butterfly effect is the phenomenon that a smal...
  Rank  7 | KB idx  168 | score=4.7072 | The butterfly effect is the phenomenon that a smal...
  Rank  8 | KB idx  576 | score=4.6870 | The butterfly effect is the phenomenon that a smal...
  Rank  9 | KB idx  663 | score=4.6602 | The butterfly effect is the phenomenon that a smal...
  Rank 10 | KB idx 1701 | score=4.6602 | The butterfly effect is the phenomenon that a smal...

In [7]:
tokenizer_bert = AutoTokenizer.from_pretrained("bert-base-uncased")
 
row_42     = train.iloc[42]
prompt_42  = str(row_42["prompt"])

emb_42 = model.encode([prompt_42], show_progress_bar=False)
emb_42 = np.array(emb_42).astype("float32")
D42, I42 = index.search(emb_42, k=5)
retrieved_idx_42 = I42[0].tolist()

docs_5      = [kb[i] for i in retrieved_idx_42]
concat_docs = " ".join(docs_5)

rag_string_42 = f"Context: {concat_docs} Question: {prompt_42}"
 
print(f"Row 42 prompt: {prompt_42[:80]}...")
print(f"RAG string (first 200 chars): {rag_string_42[:200]}...")

tokens_42    = tokenizer_bert(rag_string_42, truncation=False)
token_count  = len(tokens_42["input_ids"])
 
print(f"Total tokens (no truncation): {token_count}")

Row 42 prompt: Choose the correct answer: What are permutation-inversion groups? based on the g...
RAG string (first 200 chars): Context: Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to the center of mass, or a combinati...
Total tokens (no truncation): 216


In [8]:
true_doc_150 = kb[150]
 
rag_string_150 = f"Context: {true_doc_150} Question: {prompt_150}"
 
result_q5 = zs(rag_string_150, candidate_labels=labels_150, multi_label=False)
 
scores_dict_q5    = dict(zip(result_q5["labels"], result_q5["scores"]))
correct_score_q5  = scores_dict_q5[ans_150]
 
print(f"True document   : {true_doc_150}")
print(f"RAG string      : {rag_string_150[:120]}...")
print(f"All scores      : {dict(zip(result_q5['labels'], [round(s,4) for s in result_q5['scores']]))}")
print(f"P(correct option) with TRUE context: {round(correct_score_q5, 3)}")
print(f"Change vs Q1 (no RAG): {round(correct_score_q5 - correct_score_q1, 3)}")

True document   : The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
RAG string      : Context: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework c...
All scores      : {'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."': 0.9894, 'The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."': 0.0045, 'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical structure has no effect on

In [9]:
adversarial_doc  = kb[999]
adv_rag_string   = f"Context: {adversarial_doc} Question: {prompt_150}"
 
result_q6        = zs(adv_rag_string, candidate_labels=labels_150, multi_label=False)
scores_dict_q6   = dict(zip(result_q6["labels"], result_q6["scores"]))
correct_score_q6 = scores_dict_q6[ans_150]
 
print(f"Adversarial doc (KB 999): {adversarial_doc[:80]}...")
print(f"All scores: {dict(zip(result_q6['labels'], [round(s,4) for s in result_q6['scores']]))}")
print(f"P(correct option) with WRONG context: {round(correct_score_q6, 3)}")
print(f"Summary comparison:")
print(f"No RAG (Q1)      : {round(correct_score_q1, 3)}")
print(f"True RAG (Q5)    : {round(correct_score_q5, 3)}")
print(f"Adversarial (Q6) : {round(correct_score_q6, 3)}")

Adversarial doc (KB 999): A thought experiment in which a demon guards a microscopic trapdoor in a wall se...
All scores: {'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."': 0.5289, 'The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."': 0.4253, 'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical structure has no effect on subsequent states, as defined by Lorenz in his book "The Essence of Chaos."': 0.0204, 'The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical framework has no effect on subsequent states, as defined by Lorenz in his book "The Essence 

In [10]:
hits = 0
 
for i in range(100):
    row = train.iloc[i]
    prompt_i = str(row["prompt"])
    correct_i = str(row[row["answer"]])
 
    emb_i= model.encode([prompt_i], show_progress_bar=False)
    emb_i= np.array(emb_i).astype("float32")
    _, I_i= index.search(emb_i, k=5)
    retrieved_docs = [kb[j] for j in I_i[0]]

    if any(correct_i in doc for doc in retrieved_docs):
        hits += 1
 
hit_rate = (hits / 100) * 100
print(f"Hits: {hits} / 100")
print(f"Hit Rate: {round(hit_rate, 1)}%")

Hits: 73 / 100
Hit Rate: 73.0%


In [11]:
def map_at_3(ground_truth_text, ranked_labels, label_to_text):
    correct_letter = None
    for letter, text in label_to_text.items():
        if text == ground_truth_text:
            correct_letter = letter
            break
 
    for rank, letter in enumerate(ranked_labels[:3], start=1):
        if letter == correct_letter:
            return 1.0 / rank
    return 0.0

In [12]:
map3_scores = []
CHOICES = ["A", "B", "C", "D", "E"]
 
print("Running full RAG pipeline for rows 0–19...\n")
 
for i in range(20):
    row = train.iloc[i]
    prompt_i = str(row["prompt"])
    options_i = {c: str(row[c]) for c in CHOICES}
    answer_i  = str(row[row["answer"]])
 
    emb_i = model.encode([prompt_i], show_progress_bar=False)
    emb_i = np.array(emb_i).astype("float32")
    _, I_i = index.search(emb_i, k=5)
    docs_5_i = [kb[j] for j in I_i[0]]

    pairs_i = [[prompt_i, doc] for doc in docs_5_i]
    ce_scores_i = cross_encoder.predict(pairs_i)
    best_doc_i = docs_5_i[int(np.argmax(ce_scores_i))]

    rag_i = f"Context: {best_doc_i} Question: {prompt_i}"

    candidate_texts_i = list(options_i.values())
    result_i = zs(rag_i, candidate_labels=candidate_texts_i, multi_label=False)

    text_to_letter = {v: k for k, v in options_i.items()}
    ranked_letters = [text_to_letter[lbl] for lbl in result_i["labels"]]

    score_i = map_at_3(answer_i, ranked_letters, options_i)
    map3_scores.append(score_i)
 
    print(f"  Row {i:2d} | correct={row['answer']} | "
          f"top3={ranked_letters[:3]} | MAP@3={score_i:.3f} | "
          f"best_doc={best_doc_i[:40]}...")
 
avg_map3 = np.mean(map3_scores)
print(f"Average MAP@3 (rows 0–19): {round(avg_map3, 3)}")
print(f"Per-row scores: {[round(s, 3) for s in map3_scores]}")

Running full RAG pipeline for rows 0–19...

  Row  0 | correct=B | top3=['B', 'D', 'A'] | MAP@3=1.000 | best_doc=Martin Heidegger believes that humans do...
  Row  1 | correct=A | top3=['A', 'E', 'C'] | MAP@3=1.000 | best_doc=Accelerator-based light-ion fusion is a ...
  Row  2 | correct=C | top3=['C', 'D', 'B'] | MAP@3=1.000 | best_doc=The matter and radiation that exist in t...
  Row  3 | correct=B | top3=['B', 'D', 'A'] | MAP@3=1.000 | best_doc=Martin Heidegger believes that humans do...
  Row  4 | correct=A | top3=['A', 'B', 'C'] | MAP@3=1.000 | best_doc=Simultaneity is relative, meaning that t...
  Row  5 | correct=C | top3=['B', 'C', 'A'] | MAP@3=0.500 | best_doc=The Josephson effect is a phenomenon exp...


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  Row  6 | correct=E | top3=['E', 'B', 'D'] | MAP@3=1.000 | best_doc=A pattern left by a particle-laden liqui...
  Row  7 | correct=A | top3=['A', 'B', 'C'] | MAP@3=1.000 | best_doc=The Liouville density is a probability d...
  Row  8 | correct=A | top3=['A', 'C', 'D'] | MAP@3=1.000 | best_doc=Distinguishing background particles from...
  Row  9 | correct=A | top3=['A', 'B', 'C'] | MAP@3=1.000 | best_doc=The Wigner function W(x, p) is the Wigne...
  Row 10 | correct=C | top3=['C', 'A', 'D'] | MAP@3=1.000 | best_doc=The Peierls bracket is a Poisson bracket...
  Row 11 | correct=B | top3=['B', 'C', 'A'] | MAP@3=1.000 | best_doc=The throttling process is a steady adiab...
  Row 12 | correct=D | top3=['D', 'A', 'B'] | MAP@3=1.000 | best_doc=The angular spacing of features in the d...
  Row 13 | correct=E | top3=['E', 'D', 'B'] | MAP@3=1.000 | best_doc=Supersymmetry is an intrinsic property o...
  Row 14 | correct=E | top3=['E', 'A', 'D'] | MAP@3=1.000 | best_doc=Minkowski space is a mathem